# **Laboratorio #1 - Aprendizaje por Refuerzo**
* Paula Barillas - 22764
* Gerardo Pineda - 22880
* Monica Salvatierra - 22249
* Bianca Calderón - 22272

Link del repositorio: https://github.com/paulabaal12/LAB1-RL

## **Task 1: Diseño formal del MDP**

**Contexto:** Una empresa de logística urbana opera una flota de drones de reparto en una ciudad modelada como una cuadrícula de 5x5. Cada dron parte de una base central, debe entregar paquetes en puntos designados y regresar. La empresa quiere entender si es posible modelar este problema de despacho como un MDP antes de invertir en un sistema de RL completo.

Su grupo ha sido contratado como equipo de consultoría técnica para diseñar y analizar el MDP que representa este problema, implementar un evaluador de políticas desde cero y producir un reporte técnico con conclusiones accionables.


## **1. Espacio de estados 𝒮**

En un MDP, el estado debe contener la información suficiente para que la elección de la siguiente acción dependa solo del estado actual y no de todo el historial previo, condición conocida como propiedad de Markov. En este problema, el estado debe incluir información que permita decidir si el drone debe seguir entregando, regresar a la base o evitar zonas peligrosas. Esta idea es consistente con la formulación clásica de MDPs, donde el estado representa la información relevante para la toma de decisiones futuras [1, 2].

Un estado útil para este problema es:

$$
 s = \left( p_t, d_t, b_t, c_t, h_t \right)
$$

donde:
- $p_t = (x_t, y_t)$: posición actual del drone en la cuadrícula 5x5.
- $d_t \in \{0,1,2,\dots,n\}$: número de entregas pendientes o índice del siguiente destino.
- $b_t \in [0,1]$: nivel de batería actual, normalizado.
- $c_t \in \{0,1\}$: indicador de si el drone está cargado o no en la base.
- $h_t \in \{0,1\}$: indicador de si el entorno presenta condiciones de vuelo seguras o no.

### Justificación de las variables
- La posición $p_t$ es necesaria porque las transiciones y la recompensa dependen del movimiento; este es un supuesto estándar en los MDPs, donde el estado captura la situación actual del sistema [1].
- El estado de entregas $d_t$ es esencial para saber si el drone debe continuar entregando o regresar a la base, ya que la política debe adaptarse al volumen de trabajo pendiente [2].
- La batería $b_t$ es crítica porque el drone no puede operar indefinidamente y debe considerar la viabilidad de continuar; en RL y control óptimo, los recursos limitados suelen incorporarse explícitamente en el estado para evitar decisiones miope [2].
- El indicador $c_t$ permite distinguir si el drone está listo para salir de la base o si ya ha iniciado su ruta, lo cual afecta la disponibilidad de acciones en cada instante.
- La variable $h_t$ representa un factor externo de seguridad, útil para modelar restricciones operativas. En problemas reales, la seguridad y las restricciones de operación suelen introducir variables adicionales en el estado o en la estructura del entorno [2].

### Variables omitidas y consecuencias
Se puede omitir el historial completo de rutas anteriores, siempre que el estado incluya la posición, las entregas pendientes y la batería. Este tipo de simplificación es común en MDPs porque el estado debe ser suficientemente informativo, pero no necesariamente incluir todo el pasado [1]. Si se omitiera la batería, el modelo podría tomar decisiones que conduzcan a fallas de energía. Si se omitiera el estado de entregas, el agente podría repetir rutas innecesarias o no completar la misión, lo cual reduce la calidad de la política aprendida [2].

### Definición compacta del espacio de estados

a) El estado es discreto, porque la posición y los indicadores son variables discretas, mientras que la batería puede tratarse como un valor discretizado en niveles.

b) Un espacio de estados razonable es:

$$
\mathcal{S} = \{(x,y,d,b,c,h)\}
$$

con $x,y \in \{0,1,2,3,4\}$, $d \in \{0,1,\dots,n\}$, $b \in \{0,0.25,0.5,0.75,1\}$, $c \in \{0,1\}$ y $h \in \{0,1\}$.

## **2. Espacio de acciones 𝒜**

En un MDP, el espacio de acciones define las decisiones disponibles para el agente en cada estado. Para este problema, las acciones pueden modelarse como movimientos discretos y operaciones de servicio, lo cual es una forma natural de representar problemas de control secuencial en entornos finitos [1, 2].

Las acciones del drone pueden definirse como movimientos en la cuadrícula:

$$
\mathcal{A} = \{N, S, E, O,\text{esperar},\text{entregar},\text{recargar}\}
$$

### Justificación
- Los movimientos cardinales permiten modelar rutas de reparto simples y realistas.
- La acción de esperar es útil cuando el drone debe detenerse por congestión o seguridad.
- La acción de entregar es necesaria para completar el servicio en una ubicación objetivo.
- La acción de recargar permite representar la necesidad de volver a la base o a una estación de carga.

### Restricciones en ciertos estados
No todas las acciones son válidas en todo momento. Este tipo de restricciones es natural en MDPs y puede interpretarse como un conjunto de acciones disponibles $\mathcal{A}(s)$ que depende del estado, una formulación ampliamente usada en problemas de control y planificación [1].

- Si el drone está en la base y no hay entregas pendientes, la acción de entregar no aplica.
- Si el drone está en una celda de la cuadrícula y el movimiento llevaría fuera de los límites, esa acción debe estar prohibida.
- Si la batería es muy baja, la acción de recargar debe ser priorizada o, en algunos modelos, prohibida si ya está en la base.

Estas restricciones pueden modelarse de dos formas:
1. Como una función de validez $\text{valid}(s,a)$ que retorna verdadero si la acción es permitida en el estado $s$.
2. Como un MDP con acciones restringidas, donde las acciones no válidas se asignan probabilidad cero.

En este diseño se usa la segunda opción: las acciones inválidas quedan fuera del soporte de la transición.

## **3. Función de recompensa $r(s,a,s')$**

La recompensa es uno de los elementos centrales del MDP, porque define qué comportamiento se desea promover. En RL, una recompensa mal diseñada puede inducir políticas que optimicen un criterio incorrecto o incluso comportamientos indeseados, por lo que su diseño debe alinearse con los objetivos operativos del sistema [2].

La recompensa debe reflejar el objetivo real de la empresa, no solo la finalización rápida. Se propone una función compuesta con tres componentes:

$$
 r(s,a,s') = w_e \cdot r_{entrega} + w_b \cdot r_{bateria} + w_s \cdot r_{seguridad}
$$

### Componentes propuestos
- $r_{entrega}$: recompensa positiva por completar una entrega y una penalización por cada paso de viaje sin progreso.
- $r_{bateria}$: penalización por consumo de batería y recompensa por regresar a la base con carga suficiente.
- $r_{seguridad}$: penalización fuerte si el drone entra en zonas restringidas, realiza maniobras peligrosas o vuela bajo condiciones inseguras.

### Ponderación sugerida
Se sugiere:
- $w_e = 0.5$
- $w_b = 0.3$
- $w_s = 0.2$

Esta ponderación prioriza la entrega, pero no ignora la sostenibilidad operativa ni el riesgo. En términos de RL, este tipo de trade-off entre objetivos múltiples es precisamente el tipo de problema que suele abordarse mediante recompensas compuestas o métodos de optimización multiobjetivo [2]. Si se sobrepondera la eficiencia, el agente podría ignorar la batería y volar en zonas inseguras. Si se sobrepondera la batería, el agente podría quedarse demasiado tiempo esperando o regresar prematuramente, reduciendo el desempeño de entregas. Si se subestima la seguridad, el sistema podría aprender políticas peligrosas.

### Ejemplo de recompensa concreta
- $+100$ por entrega exitosa.
- $-1$ por cada paso de movimiento.
- $-10$ por consumo de batería significativo.
- $-50$ por entrar en zona insegura o violar una restricción.
- $+20$ por regresar a la base con batería suficiente.

## **4. Función de transición $p(s' \mid s,a)$**

La transición es el componente que describe cómo evoluciona el sistema al ejecutar una acción. En MDPs, esta función puede ser determinista o estocástica; en problemas reales, lo más común es que sea estocástica, porque el resultado de una acción suele verse afectado por ruido, fallas o condiciones cambiantes del entorno [1].

La transición puede modelarse como estocástica, porque en la práctica existen incertidumbres de entorno que afectan el resultado del movimiento. Se propone un modelo simple pero realista:

### Fuentes de aleatoriedad
- Fallos o desviaciones de trayectoria por viento o interferencia.
- Congestión o bloqueo parcial de la ruta.
- Variación en la eficiencia de la batería según la carga y el clima.

### Transiciones concretas
Supóngase que el drone intenta moverse al norte desde una celda no límite.

1. Con probabilidad $0.8$, llega correctamente a la celda objetivo.
2. Con probabilidad $0.15$, se desplaza a una celda lateral por error de navegación.
3. Con probabilidad $0.05$, no cambia de posición porque el movimiento no se ejecuta correctamente.

Otra transición posible es al intentar entregar:
1. Con probabilidad $0.9$, la entrega se completa correctamente.
2. Con probabilidad $0.1$, la entrega falla temporalmente y el paquete permanece pendiente.

Y al intentar recargar:
1. Con probabilidad $0.95$, la batería aumenta significativamente.
2. Con probabilidad $0.05$, la recarga no tiene efecto por fallo del sistema.

Estas probabilidades reflejan que el sistema no es totalmente determinista, aunque se mantiene un comportamiento dominante. Este tipo de modelado es coherente con la definición clásica de un MDP estocástico, donde la transición asigna probabilidades a los posibles estados siguientes [1].

## **5. Factor de descuento $\gamma$**

El factor de descuento indica cuánto pesa el futuro frente a las recompensas inmediatas. En MDPs y RL, un valor de descuento alto implica que las decisiones futuras siguen siendo relevantes, mientras que un valor bajo hace que el agente sea más miope [1, 2].

Se propone un valor de:

$$
\gamma = 0.95
$$

### Justificación
Este valor es apropiado porque el problema combina decisiones de corto plazo y planificación de mediano plazo. Un descuento alto refleja que las entregas futuras siguen siendo valiosas, pero no tanto como las ganancias inmediatas. Si el descuento fuera demasiado alto, el agente se concentraría excesivamente en recompensas inmediatas y no planificaría rutas robustas. Si fuera demasiado bajo, el agente priorizaría demasiado el futuro, lo que puede ser poco práctico para operaciones diarias donde la respuesta rápida es importante. Esta interpretación es consistente con la noción general de retorno descontado en MDPs y RL [1, 2].

## **Task 2**

**1. Identifiquen al menos dos situaciones en las que la propiedad de Markov se violaría con su representación de estado actual. Para cada una, propongan una extensión del estado que restaure la propiedad y discutan el costo computacional de esa extensión.**

En primer lugar, la representación actual del estado incumple la propiedad de Markov en lo referente a la batería, dado que la variable 𝑏𝑡 registra únicamente el nivel de carga presente, sin reflejar el desgaste acumulado por ciclos de uso. En consecuencia, dos estados con 𝑏𝑡 idéntico pero distinto historial de operación tendrían tasas de descarga futuras diferentes, lo cual implica que la transición depende de información externa al estado actual. Para restaurar la propiedad, se propone incorporar una variable adicional de salud o desgaste de batería 𝑒𝑡; sin embargo, esto incrementaría el espacio de estados, puesto que corresponde al producto cartesiano de los valores posibles de cada variable. Agregar una nueva variable con 𝑘 valores posibles multiplica por 𝑘 el tamaño total del espacio de estados, según el nivel de estados considerados.

Asimismo, ocurre una violación similar con la variable climática ℎ𝑡, ya que esta se modela como independiente en cada paso, cuando en realidad las condiciones meteorológicas presentan correlación temporal, de modo que un evento inseguro tiende a persistir durante varios pasos consecutivos. Restaurar la propiedad de Markov requeriría añadir un contador de pasos consecutivos inseguros o una variable de tendencia climática, lo que también aumenta el tamaño del espacio de estados, aunque en menor medida que una ventana completa de historial.



**2. En el dominio de logística urbana, ¿qué tan razonable es el supuesto de que el agente puede observar el estado completo? ¿Qué variables del estado real probablemente no son directamente observables por el drone? ¿Cómo afecta eso la validez del modelo MDP versus un modelo POMDP?**

El supuesto de observabilidad completa resulta poco sostenible al analizar el dominio de logística urbana con mayor detalle. Esto se debe a que varias variables relevantes del sistema real no son accesibles de forma directa para el dron, entre ellas la posición exacta, dado que los sistemas de posicionamiento satelital introducen ruido. Asimismo, la salud real de la batería, ya que el indicador de carga es apenas un proxy que no captura el efecto del desgaste ni de la temperatura y las condiciones exactas de viento u obstáculos dinámicos no detectados oportunamente. 

Por ello, el problema se ajusta con mayor fidelidad a la estructura de un POMDP, en el cual el agente actúa sobre observaciones parciales y ruidosas de un estado oculto, en lugar de sobre el estado verdadero. No obstante, en la práctica suele adoptarse una aproximación intermedia que consiste en tratar el problema como un MDP definido sobre el estado estimado, por ejemplo mediante un filtro de Kalman. Utilizar esta simplificación es razonable cuando el error de estimación es reducido, pero puede llevar a políticas subóptimas cuando la incertidumbre del entorno es considerable.


**3. Argumenten si este problema debería modelarse como una tarea episódica o continua. ¿Cómo cambia esa decisión el diseño de la función de recompensa y el valor de 𝛾?**

Resulta más adecuado modelarlo como una tarea episódica, puesto que la operación presenta un ciclo delimitado con estados terminales claramente identificables. Por ejemplo, el retorno exitoso a la base, por un lado, y el fallo por agotamiento de batería o colisión, por otro. Esta elección tiene implicaciones directas sobre el diseño de la función de recompensa y sobre el valor de 𝛾, ya que, al tratarse de un horizonte finito, el factor de descuento puede aproximarse a uno sin riesgo de divergencia, y las recompensas terminales pueden emplearse de forma natural para reforzar tanto el cumplimiento de la entrega como la seguridad del vuelo. 

En cambio, si el problema se planteara como una tarea continua, correspondiente a la operación ininterrumpida de la flota a lo largo del día, sería necesario adoptar un valor de 𝛾 estrictamente menor a uno o recurrir a una formulación de recompensa promedio, transformando los incentivos terminales en recompensas recurrentes distribuidas a lo largo del proceso.

## **Task 3**

## **Task 4**

## **Fuentes consultadas**

1. Puterman, M. L. (1994). Markov Decision Processes. Wiley.
2. Sutton, R. S., & Barto, A. G. (2018). Reinforcement Learning: An Introduction (2nd ed.). MIT Press.
3. Wikipedia: Markov decision process y Reinforcement learning, como referencia general para conceptos de base.